Here we first call the imports and we mount the google drive to be able to load our data files.

In [19]:
# Imports
import pandas as pd
import pickle
from collections import defaultdict
import numpy as np
from google.colab import drive
!pip install stellargraph
!pip install gensim==4.0
import stellargraph as sg
from stellargraph.data import BiasedRandomWalk
from stellargraph.data import UniformRandomMetaPathWalk
import gensim
from pylab import *
from sklearn.manifold import TSNE
from gensim.models import Word2Vec
import random
!pip install networkx
import networkx as nx
import operator

Please change the below part if needed to run the data files accordingly

In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
data_path = "/content/drive/MyDrive/Projects/UALR/Spring 2021/Data Science and Technologies/Final Project/Code/MERGE/"

Mounted at /content/drive


I already have the PubMed dataset introduced in this article https://www.nature.com/articles/s41597-020-0543-2. The dataset is a MySQL database of the full PubMed library in addition to extracted and disambiguiated authors. Here however we load all the data avaialble even the preprocessed ones.

In [3]:
articles_df = pd.read_csv(data_path+"A01_Articles_Top_50_Cancer.csv", sep=",", encoding='utf-8')
abstracts_df = pd.read_csv(data_path+"A04_Abstract_Top_50_Cancer.csv", sep=",", encoding='utf-8')
bioentities_df = pd.read_csv(data_path+"B10_BERN_Main_Top_50_Cancer.csv", sep=",", encoding='utf-8')
article_author_dataframe = pd.read_csv(data_path+"A02_AuthorList_Cancer_50.csv", sep=",", encoding='utf-8', usecols=["PMID","AID"])
citations_dataframe = pd.read_csv(data_path+"B15_WOS_Citation_Top_50_Cancer_Inter.csv", sep=",", encoding='utf-8', usecols=["PMID","RefArticleId"])
c2b2r_node_types = pd.read_csv(data_path+"node2class.txt", sep="\t", encoding='utf-8', header=None)
drug_mapping = pd.read_csv(data_path+"drug-list.txt", sep="\t", encoding='utf-8', dtype=object)
gene_mapping = pd.read_csv(data_path+"gene-list.txt", sep="\t", encoding='utf-8', dtype=object)
chem2bio2rdf_dataset = pd.read_csv(data_path+"chem2bio2rdf.txt", sep="\t", encoding='utf-8', header=None)
c2b2r_node_types = pd.read_csv(data_path+"node2class.txt", sep="\t", encoding='utf-8', header=None)
print(articles_df.info())
print(abstracts_df.info())
print(bioentities_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 39 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   PMID                                      50 non-null     int64  
 1   PMID_Version                              50 non-null     int64  
 2   MedlineCitation_Owner                     50 non-null     object 
 3   MedlineCitation_Status                    50 non-null     object 
 4   DateCompleted                             50 non-null     object 
 5   DateRevised                               50 non-null     object 
 6   Article_PubModel                          50 non-null     object 
 7   MedLinePgn                                50 non-null     object 
 8   ElocationID                               22 non-null     object 
 9   ArticalDate                               21 non-null     object 
 10  Language                                

First we prepare the Pubmed abstracts by looping on them and resolving any ambiguity. Some articles have mulitple abstracts. The output of this step is a dictionary of articles PMID and abstracts

In [4]:
corpus = {}
for index,abstract in articles_df.iterrows():
    pmid = abstract[0]
    record = abstracts_df.loc[abstracts_df['PMID'] == pmid]
    abstract_extracted = ""
    for index2,abstract_part in record.iterrows():
        abstract_extracted = abstract_extracted + abstract_part[2]
    if len(abstract_extracted) != 0:
        corpus[int(pmid)] = str(abstract_extracted)
print(corpus) 

{7553621: "The tumor suppressor gene CDKN2/p16/MTS1, located on chromosome 9p21, is frequently inactivated in many human cancers through homozygous deletion. Recently, we have reported another pathway of inactivation that involves loss of transcription associated with de novo methylation of a 5' CpG island of CDKN2/p16 in lung cancers, gliomas, and head and neck squamous cell carcinomas. We now show that this aberrant CpG island methylation also occurs frequently in cell lines of breast cancer (33%), prostate cancer (60%), renal cancer (23%), and colon cancer (92%) and is associated with loss of transcription. Primary tumors of the breast (31%) and colon (40%) also displayed de novo methylation of this CpG island. This alteration of p16 in colon cancer was particularly striking, since inactivation does not occur through homozygous deletion in this tumor type. Our data show that in tumors, de novo methylation of the 5' CpG island is a frequent mode of inactivation of CDKN2/p16 and also 

Next we extract the BioBERT entities already present in the PubMed dataset. The ouput of this step is a dictionary of unique bioentities ids and a list of their different mentions extracted from all the articles. Also a dictionary for each PMID and the Entity ids that were extracted from it.

In [5]:
print("Start BioBERT Bioentity Extraction")
bioentities_articles_dictionary = {}
unique_bioentities_dictionary = {}
for pmid in corpus:
    extracted_bioentities_df = bioentities_df[bioentities_df["PMID"] == pmid]
    bioentities_in_articles_dict = {}
    sorted_bioentities = {}
    for be in extracted_bioentities_df.iterrows():
        bioentity_id = str(be[1]["EntityID"])
        bioentity_type = str(be[1]["Type"])
        bioentity_mention = str(be[1]["Mention"])
        if (bioentity_id != "0") and (bioentity_type in ["2", "3", "4"]):
            typed_bioentity_id = ""
            if bioentity_type == "2":
                typed_bioentity_id = str("disease/" + str(bioentity_id))
            if bioentity_type == "3":
                typed_bioentity_id = str("gene/" + str(bioentity_id))
            if bioentity_type == "4":
                typed_bioentity_id = str("drug/" + str(bioentity_id))
            bioentities_in_articles_dict[bioentity_mention] = typed_bioentity_id
            # Fillup bioentity id
            unique_bioentities_dictionary[bioentity_mention] = typed_bioentity_id
        sorted_bioentities = defaultdict(list)
        for key, val in sorted(bioentities_in_articles_dict.items()):
            sorted_bioentities[val].append(key)
        sorted_bioentities = dict(sorted_bioentities)

    bioentities_articles_dictionary[str(pmid)] = sorted_bioentities

sorted_unique_bioentities = defaultdict(list)
for key, val in sorted(unique_bioentities_dictionary.items()):
    sorted_unique_bioentities[val].append(key)
sorted_unique_bioentities = dict(sorted_unique_bioentities)

print(sorted_unique_bioentities)

Start BioBERT Bioentity Extraction
{'gene/324006902': ['-mir-155', 'hsa-mir-155'], 'drug/4660903': ['1,3-bis(2-chloroethyl)-1-nitrosourea', 'carmustine'], 'drug/282474603': ['2-hydroxyglutarate'], 'drug/4731903': ["5-aza-2'-deoxycytidine", "5-aza-2'deoxycytidine", '5-deoxyazacytidine'], 'drug/276915103': ['5-methylcytosine'], 'disease/106939701': ['AIS', 'adenocarcinoma in situ'], 'disease/107405101': ['ALL', 'acute lymphoblastic leukemia'], 'disease/107418901': ['AML', 'AMLs', 'acute leukemias', 'acute myeloid leukemia'], 'gene/323620702': ['APC'], 'gene/325260002': ['ARID1A'], 'disease/106965901': ['Adenocarcinoma', 'adenocarcinoma', 'adenocarcinomas'], 'disease/256012401': ['Adenocarcinoma of the lung', 'lung adenocarcinoma', 'lung adenocarcinomas'], 'disease/258108801': ['B-cell malignancies'], 'disease/260797101': ['BAC'], 'gene/324888402': ['BRCA1'], 'gene/324891202': ['BRCA2'], 'gene/325409102': ['CCNE1'], 'gene/323564402': ['CD274', 'PD-L1'], 'gene/325552102': ['CDH1', 'E-cadhe

Next we map the extracted entities to the Chem2Bio2RDF dataset. Note that there is a manual part that I had to do with mapping drug and gene names with their normailized entity id to align them with the candidate entities in Chem2Bio2RDF.

In [6]:
mapping_drug_biobert_entity_id_to_c2b2r = []
mapping_gene_biobert_entity_id_to_c2b2r = []
for index, r1 in drug_mapping.iterrows():
    drug_mention = str(r1[0])
    pubchem_id = str(r1[1])
    for be1 in sorted_unique_bioentities:
        list_of_mentions = sorted_unique_bioentities[be1]
        if drug_mention in list_of_mentions:
            # then map entity to c2b2r
            c2b2r_string_1 = "http://chem2bio2rdf.org/pubchem/resource/pubchem_compound/" + pubchem_id
            mapping_drug_biobert_entity_id_to_c2b2r.append((be1, c2b2r_string_1))
for index, r2 in gene_mapping.iterrows():
    gene_mention = str(r2[0])
    for be2 in sorted_unique_bioentities:
        list_of_mentions_2 = sorted_unique_bioentities[be2]
        if gene_mention in list_of_mentions_2:
            # then map entity to c2b2r
            c2b2r_string_2 = "http://chem2bio2rdf.org/uniprot/resource/gene/" + gene_mention.upper()
            mapping_gene_biobert_entity_id_to_c2b2r.append((be2, c2b2r_string_2))
# Search for the entities in chem2bio2rdf if found then we keep the mapping record, else we remove it from the mapping
mapping_biobert_entity_id_to_c2b2r = mapping_drug_biobert_entity_id_to_c2b2r + mapping_gene_biobert_entity_id_to_c2b2r
source_nodes = []
target_nodes = []
for index, triple in chem2bio2rdf_dataset.iterrows():
    source_node = str(triple[0])
    edge_type = str(triple[1])
    target_node = str(triple[2])
    source_nodes.append(source_node)
    target_nodes.append(target_node)
unique_nodes = np.unique(np.array(source_nodes + target_nodes))
final_list = {}
for record in mapping_biobert_entity_id_to_c2b2r:
    if record[1] in unique_nodes:
        final_list[record[0]] = record[1]
print(final_list)

{'drug/4660903': 'http://chem2bio2rdf.org/pubchem/resource/pubchem_compound/2578', 'drug/282474603': 'http://chem2bio2rdf.org/pubchem/resource/pubchem_compound/43', 'drug/4731903': 'http://chem2bio2rdf.org/pubchem/resource/pubchem_compound/451668', 'drug/94137403': 'http://chem2bio2rdf.org/pubchem/resource/pubchem_compound/6322', 'drug/4249503': 'http://chem2bio2rdf.org/pubchem/resource/pubchem_compound/104748', 'drug/121768403': 'http://chem2bio2rdf.org/pubchem/resource/pubchem_compound/13711', 'drug/4148503': 'http://chem2bio2rdf.org/pubchem/resource/pubchem_compound/5962', 'drug/4140903': 'http://chem2bio2rdf.org/pubchem/resource/pubchem_compound/5994', 'drug/4284203': 'http://chem2bio2rdf.org/pubchem/resource/pubchem_compound/216239', 'drug/4255203': 'http://chem2bio2rdf.org/pubchem/resource/pubchem_compound/5394', 'drug/304491003': 'http://chem2bio2rdf.org/pubchem/resource/pubchem_compound/6057', 'drug/4215603': 'http://chem2bio2rdf.org/pubchem/resource/pubchem_compound/1174', 'ge

Now we need to create a mapping dictionary between articles and authors.

In [7]:
article_author_mapping = {}
for index, row in article_author_dataframe.iterrows():
    article = row[0]
    author = row[1]
    article_author_mapping[author] = article
sorted_article_author_mapping = defaultdict(list)
for key, val in sorted(article_author_mapping.items()):
    sorted_article_author_mapping[val].append(key)
article_author_mapping = dict(sorted_article_author_mapping)
print(article_author_mapping)

{24120142: [0, 68933, 118243, 1002523, 1045079, 1094127, 1361744, 1389073, 1471632, 1720119, 2087125, 2165798, 2202853, 2228037, 2448328, 2601855, 2610069, 3422492, 4236769, 4319069, 4550703, 4552002, 4567155, 4963014, 4977372, 5223697, 5418167, 5451950, 5481511, 5784165, 5833880, 6002070, 6140037, 6163016, 6210318, 6445355, 6491831, 6590752, 6948774, 7023938, 7244269, 7261593, 7540541, 8497010, 8515702, 8534108, 8644316, 8829450, 9069143, 9264633, 16317263, 20846630, 23432293, 23550987, 23914279, 24359617, 24374471, 24472933, 24757214], 15328174: [23415, 171031, 177401, 606392, 1343395, 1371532, 1675051, 1908560, 3757672, 4485404, 5493960, 6552802, 8555429, 8644883, 9214945], 23634996: [26194, 58831, 162106, 167846, 216805, 229295, 241590, 309342, 392109, 680794, 1391984, 1413524, 2213339, 2383541, 2476917, 2483355, 2934347, 2934754, 2962823, 3180722, 3336524, 3344031, 3571757, 3684640, 4057161, 4288717, 4417050, 4875476, 4910425, 5986259, 6133954, 6139860, 6188913, 6236361, 6543827, 

We also need to create a mapping between articles and citations.

In [8]:
citations = {}
for index, row in citations_dataframe.iterrows():
    source_article = str(row[0])
    target_article = str(row[1])
    citations[source_article] = target_article
print(citations)

{'7585152': '7937876', '7553621': '7937876', '8790415': '7553621', '9041175': '8790415', '9618505': '7937876', '9916800': '9618505', '10411935': '9041175', '11070098': '8790415', '10676951': '10521349', '11707567': '10676951', '11309270': '9618505', '16007088': '10411935', '15765097': '11309270', '16530703': '15172979', '17157791': '15328174', '16804544': '8790415', '16989803': '15172979', '17334365': '16007088', '18772890': '11070098', '18600261': '17334365', '19151715': '17334365', '19269895': '11070098', '20129251': '18772890', '21130701': '20399149', '20399149': '11070098', '21252716': '16530703', '21720365': '18772890', '22960745': '21720365', '22810696': '18772890', '22588877': '21720365', '24132290': '18772890', '23636398': '10411935', '24120142': '22960745', '23634996': '21130701', '25079317': '23636398', '25079552': '11707567'}


Next is the curcial step of entity alignment between the two graphs and creating a larger graph data structure that holds all the integrated data together

In [9]:
print("Constructing PubMed Graph")
graph = []
bioentity_c2b2r_mapping = final_list
article_bioentity_mapping = bioentities_articles_dictionary
citation_mapping = citations
linked_bioentities = []
linked_articles = []
chem2bio2rdf_entities = []
c2b2r_intermediate_mapping = {}
for be in bioentity_c2b2r_mapping:
    c2b2r_entity = bioentity_c2b2r_mapping[be]
    source_node = "bioentity/" + str(be)
    relation = "sameAs"
    target_node = str(c2b2r_entity)
    linked_bioentities.append(source_node)
    chem2bio2rdf_entities.append(target_node)
    c2b2r_intermediate_mapping[target_node] = source_node

for pmid in article_bioentity_mapping:
    bioentities_dict = article_bioentity_mapping[pmid]
    for be in bioentities_dict.keys():
        source_node = "bioentity/"+str(be)
        relation = "mentionedIn/mentions"
        target_node = "article/pmid/"+str(pmid)
        # if bioentity is linked to chem2bio2rdf
        if source_node in linked_bioentities:
            linked_articles.append(target_node)
            graph.append((source_node,relation,target_node))

for pmid in article_author_mapping:
    authors_dict = article_author_mapping[pmid]
    for a in authors_dict:
        source_node = "author/"+str(a)
        relation = "wrote/isWrittenBy"
        target_node = "article/pmid/"+str(pmid)
        if target_node in linked_articles:
            graph.append((source_node,relation,target_node))

for pmid in citation_mapping:
    linked_article = citation_mapping[pmid]
    source_node = "article/pmid/" + str(pmid)
    relation = "cites/citedBy"
    target_node = "article/pmid/" + str(linked_article)
    if (source_node in linked_articles) or (target_node in linked_articles):
        graph.append((source_node, relation, target_node))

# Link chem2bio2rdf
unwanted_relations = ["http://chem2bio2rdf.org/chebi", "http://chem2bio2rdf.org/chemogenomics/resource/expression",
                      "http://chem2bio2rdf.org/chemogenomics/resource/expression", "http://chem2bio2rdf.org/chebi",
                      "http://chem2bio2rdf.org/uniprot/resource/GO_ID", "http://chem2bio2rdf.org/substructure",
                      "http://chem2bio2rdf.org/hgnc/resource/Gene_Family_Name", "http://chem2bio2rdf.org/substructure",
                      "http://chem2bio2rdf.org/hprd/resource/tissue", "http://chem2bio2rdf.org/kegg/resource/protein"]

c2b2r_source_nodes = []
c2b2r_target_nodes = []
for index, row in chem2bio2rdf_dataset.iterrows():
    c2b2r_source_nodes.append(row[0])
    c2b2r_target_nodes.append(row[2])

for row2 in chem2bio2rdf_entities:
    res_list = []
    if (row2 in c2b2r_source_nodes):
        res_list = [i for i in range(len(c2b2r_source_nodes)) if c2b2r_source_nodes[i] == row2]
    if (row2 in c2b2r_target_nodes):
        res_list = [i for i in range(len(c2b2r_target_nodes)) if c2b2r_target_nodes[i] == row2]
    selected_c2b2r_dataframe = chem2bio2rdf_dataset.iloc[res_list, :]
    for i,r in selected_c2b2r_dataframe.iterrows():
        if (r[0] in chem2bio2rdf_entities) and (r[2] in chem2bio2rdf_entities) and (r[1] not in unwanted_relations):
            graph.append((str(c2b2r_intermediate_mapping[r[0]]),str(r[1]),str(c2b2r_intermediate_mapping[r[2]])))
        elif (r[0] in chem2bio2rdf_entities) and (r[1] not in unwanted_relations):
            graph.append((str(c2b2r_intermediate_mapping[r[0]]),str(r[1]),str(r[2])))
        elif (r[2] in chem2bio2rdf_entities) and (r[1] not in unwanted_relations):
            graph.append((str(r[0]),str(r[1]),str(c2b2r_intermediate_mapping[r[2]])))
        elif r[1] not in unwanted_relations:
            graph.append((str(r[0]), str(r[1]), str(r[2])))

Constructing PubMed Graph


Now we need to convert the graph triples to a graph object using a library called stellar graph. But before we have to do preprocessing to label the relationships between the entities.

In [10]:
source_nodes = []
target_nodes = []
orientation = []
for triple in graph:
    source_nodes.append(triple[0])
    target_nodes.append(triple[2])
    orientation.append(triple[1])
all_nodes = source_nodes + target_nodes
nodes = np.unique(np.array(all_nodes)).tolist()
graph_metadata = {}
print(str(len(nodes)))
for n in nodes:
    parts = n.split("/")
    node_type = "None"
    if parts[0] == "http:":
        found_df = c2b2r_node_types.loc[c2b2r_node_types[0] == n]
        found_node_type = found_df.iloc[0][1]
        if found_node_type == "Target":
            node_type = "gene"
        if found_node_type == "Disease":
            node_type = "disease"
        if found_node_type == "Chemical Compound/Drug":
            node_type = "drug"
        if found_node_type == "Side effect":
            node_type = "side_effect"
    if parts[0] == "article":
        node_type = "article"
    if parts[0] == "author":
        node_type = "author"
    if parts[0] == "bioentity":
        if parts[1] == "drug":
            node_type = "drug"
        if parts[1] == "gene":
            node_type = "gene"
    graph_metadata[n] = node_type
print(str(len(graph_metadata)))
print(graph_metadata) 

6731
6731
{'article/pmid/10411935': 'article', 'article/pmid/11070098': 'article', 'article/pmid/11309270': 'article', 'article/pmid/11707567': 'article', 'article/pmid/12374981': 'article', 'article/pmid/12809602': 'article', 'article/pmid/15328174': 'article', 'article/pmid/15758010': 'article', 'article/pmid/15765097': 'article', 'article/pmid/16007088': 'article', 'article/pmid/16530703': 'article', 'article/pmid/16804544': 'article', 'article/pmid/17157791': 'article', 'article/pmid/17334365': 'article', 'article/pmid/17512414': 'article', 'article/pmid/18600261': 'article', 'article/pmid/18772890': 'article', 'article/pmid/19269895': 'article', 'article/pmid/20129251': 'article', 'article/pmid/20393566': 'article', 'article/pmid/20399149': 'article', 'article/pmid/21130701': 'article', 'article/pmid/21252716': 'article', 'article/pmid/21720365': 'article', 'article/pmid/22522925': 'article', 'article/pmid/22588877': 'article', 'article/pmid/22810696': 'article', 'article/pmid/229

Now that we have some graph metadata that we will use when converting the graph triples to a graph object to perform embedding and further analysis.

In [11]:
source_nodes = []
target_nodes = []
orientation = []
for triple in graph:
    source_nodes.append(triple[0])
    target_nodes.append(triple[2])
    orientation.append(triple[1])
all_nodes = source_nodes + target_nodes
nodes = np.unique(np.array(all_nodes)).tolist()
#drug, gene, author, article, other types from chem2bio2rdf
gene_list = []
drug_list = []
disease_list = []
side_effect_list = []
article_list = []
author_list = []
print("Original size: " + str(len(nodes)))
for n in nodes:
    node_type = graph_metadata[n]
    if node_type == "gene":
        gene_list.append(n)
    if node_type == "drug":
        drug_list.append(n)
    if node_type == "disease":
        disease_list.append(n)
    if node_type == "side_effect":
        side_effect_list.append(n)
    if node_type == "article":
        article_list.append(n)
    if node_type == "author":
        author_list.append(n)
full_nodes = gene_list + drug_list + article_list + author_list + disease_list + side_effect_list
print("size: " + str(len(full_nodes)))
gene_df = pd.DataFrame([], index=gene_list)
drug_df = pd.DataFrame([], index=drug_list)
disease_df = pd.DataFrame([], index=disease_list)
side_effect_df = pd.DataFrame([], index=side_effect_list)
article_df = pd.DataFrame([], index=article_list)
author_df = pd.DataFrame([], index=author_list)
edges_df = pd.DataFrame({"source": source_nodes, "target": target_nodes, "orientation": orientation})
sg_graph = sg.StellarGraph(nodes={
                                "gene": gene_df,
                                "drug": drug_df,
                                "disease": disease_df,
                                "side_effect": side_effect_df,
                                "article": article_df,
                                "author": author_df
                        },
                        edges=edges_df,
                        edge_type_column="orientation")
print(sg_graph.info())


Original size: 6731
size: 6731
StellarGraph: Undirected multigraph
 Nodes: 6731, Edges: 7792

 Node types:
  drug: [5592]
    Features: none
    Edge types: drug-http://chem2bio2rdf.org/chemogenomics/resource/chemogenomics->gene, drug-http://chem2bio2rdf.org/omim/resource/drug->disease, drug-http://chem2bio2rdf.org/sider/resource/cid->side_effect, drug-mentionedIn/mentions->article
  gene: [500]
    Features: none
    Edge types: gene-http://chem2bio2rdf.org/chemogenomics/resource/chemogenomics->drug, gene-http://chem2bio2rdf.org/hprd->gene, gene-http://chem2bio2rdf.org/omim/resource/gene->disease, gene-mentionedIn/mentions->article
  author: [474]
    Features: none
    Edge types: author-wrote/isWrittenBy->article
  disease: [70]
    Features: none
    Edge types: disease-http://chem2bio2rdf.org/omim/resource/drug->drug, disease-http://chem2bio2rdf.org/omim/resource/gene->gene
  side_effect: [52]
    Features: none
    Edge types: side_effect-http://chem2bio2rdf.org/sider/resource/ci

Now we are ready to embed the graph using Node2vec. Node2vec involves 2 steps first we sample the graph using a biased random walk strategy.

In [12]:
print("Embedding graph")
rw = BiasedRandomWalk(sg_graph)
walks = rw.run(
  nodes=list(sg_graph.nodes()),  # root nodes
  length=50,  # maximum length of a random walk
  n=2,  # number of random walks per root node
  p=0.5,  # Defines (unormalised) probability, 1/p, of returning to source node
  q=2.0,  # Defines (unormalised) probability, 1/q, for moving away from source node
)
print("Number of random walks: {}".format(len(walks)))
str_walks = [[str(n) for n in walk] for walk in walks]

Embedding graph
Number of random walks: 13462


Then we pass the sampled graph to a Word2Vec algorithm

In [13]:
model = Word2Vec(str_walks, vector_size=128, window=5, negative=7, min_count=0, sg=1, workers=8, epochs=5, sample=0)

Next we visualize and test the graph embedding vectors for our integrated data.

In [15]:
print("Analyzing model")
%matplotlib inline
# Retrieve node embeddings and corresponding subjects
node_ids = model.wv.index_to_key  # list of node IDs
node_embeddings = (
    model.wv.vectors
)  # numpy.ndarray of size number of nodes times embeddings dimensionality
node_targets = [sg_graph.node_type(node_id) for node_id in node_ids]
transform = TSNE  # PCA
trans = transform(n_components=2)
node_embeddings_2d = trans.fit_transform(node_embeddings)
# draw the points
label_map = {l: i for i, l in enumerate(np.unique(node_targets))}
print(label_map)
node_colours = [label_map[target] for target in node_targets]
color_list = []
labels = []
for n_c in node_colours:
    c = ""
    label = ""
    if n_c == 0:
        c = "red"
        label = "article"
    if n_c == 1:
        c = "green"
        label = "author"
    if n_c == 2:
        c = "cyan"
        label="disease"
    if n_c == 3:
        c = "blue"
        label="drug"
    if n_c == 4:
        c = "black"
        label="gene"
    if n_c == 5:
        c = "yellow"
        label="side_effect"
    labels.append(label)
    color_list.append(c)
classes = ['article', 'author', 'disease', 'drug', 'gene', 'side_effect']
u = np.unique(np.array(node_colours))
fig = plt.figure(figsize=(50, 50))
ax = fig.add_subplot()
for i in range(len(color_list)):
    ax.scatter(node_embeddings_2d[i, 0], node_embeddings_2d[i, 1],
                c=color_list[i], label=labels[i], alpha=0.6, s=500)
#figure out edges
edge_lines_x = []
edge_lines_y = []
edge_lines_z = []
print(model.wv.key_to_index)
for e in sg_graph.edges():
    source_node = e[0]
    target_node = e[1]
    edge_lines_x.append(node_embeddings_2d[int(model.wv.key_to_index[source_node]), 0])
    edge_lines_x.append(node_embeddings_2d[int(model.wv.key_to_index[target_node]), 0])
    edge_lines_y.append(node_embeddings_2d[int(model.wv.key_to_index[source_node]), 1])
    edge_lines_y.append(node_embeddings_2d[int(model.wv.key_to_index[target_node]), 1])
ax.plot(edge_lines_x, edge_lines_y, alpha=0.2)
selected_nodes = random.choices(node_ids, k=50)
for txt in selected_nodes:
    ax.text(node_embeddings_2d[int(model.wv.key_to_index[txt]), 0], node_embeddings_2d[int(model.wv.key_to_index[txt]), 1], txt, fontsize=16)
plt.show()

Output hidden; open in https://colab.research.google.com to view.

Now we can do further testing using similarity between nodes

Another analysis is to find the top 10 similar drugs, genes to a specific author


In [17]:
['article', 'author', 'disease', 'drug', 'gene', 'side_effect']
article_list = []
author_list = []
disease_list = []
drug_list = []
gene_list = []
side_effect_list = []
result = model.wv.most_similar(positive=["author/6190153"], topn=1000)
for record in result:
    node_id = record[0]
    node_type = sg_graph.node_type(node_id)
    if node_type == 'article':
        article_list.append(node_id)
    if node_type == 'author':
        author_list.append(node_id)
    if node_type == 'disease':
        disease_list.append(node_id)
    if node_type == 'drug':
        drug_list.append(node_id)
    if node_type == 'gene':
        gene_list.append(node_id)
    if node_type == 'side_effect':
        side_effect_list.append(node_id)
print(article_list[0:10])
print(author_list[0:10])
print(disease_list[0:10])
print(drug_list[0:10])
print(gene_list[0:10])
print(side_effect_list[0:10])

['article/pmid/20399149', 'article/pmid/11070098', 'article/pmid/21130701', 'article/pmid/19269895', 'article/pmid/8790415', 'article/pmid/7553621', 'article/pmid/22588877']
['author/4833768', 'author/4909559', 'author/1003817', 'author/4377514', 'author/23311144', 'author/52297', 'author/3299395', 'author/5444518', 'author/8628538', 'author/18749010']
['http://chem2bio2rdf.org/omim/resource/omim_disease/925', 'http://chem2bio2rdf.org/omim/resource/omim_disease/1114', 'http://chem2bio2rdf.org/omim/resource/omim_disease/9037', 'http://chem2bio2rdf.org/omim/resource/omim_disease/1305', 'http://chem2bio2rdf.org/omim/resource/omim_disease/1074', 'http://chem2bio2rdf.org/omim/resource/omim_disease/1183', 'http://chem2bio2rdf.org/omim/resource/omim_disease/1595', 'http://chem2bio2rdf.org/omim/resource/omim_disease/523']
['http://chem2bio2rdf.org/pubchem/resource/pubchem_compound/5884', 'http://chem2bio2rdf.org/pubchem/resource/pubchem_compound/972', 'http://chem2bio2rdf.org/pubchem/resource/

Also using centrality we find the most influential objects in the graph

In [22]:
G = sg_graph.to_networkx()
pr = nx.degree_centrality(G)
sorted_d = dict(sorted(pr.items(), key=operator.itemgetter(1), reverse=True))
print(sorted_d)
article_list = []
author_list = []
disease_list = []
drug_list = []
gene_list = []
side_effect_list = []
for record in sorted_d.keys():
    node_id = record
    node_type = sg_graph.node_type(node_id)
    if node_type == 'article':
        article_list.append(node_id)
    if node_type == 'author':
        author_list.append(node_id)
    if node_type == 'disease':
        disease_list.append(node_id)
    if node_type == 'drug':
        drug_list.append(node_id)
    if node_type == 'gene':
        gene_list.append(node_id)
    if node_type == 'side_effect':
        side_effect_list.append(node_id)
print(article_list[0:2])
print(author_list[0:2])
print(disease_list[0:2])
print(drug_list[0:2])
print(gene_list[0:2])
print(side_effect_list[0:2])

{'bioentity/gene/323060702': 0.3518573551263001, 'bioentity/gene/323098302': 0.1539375928677563, 'bioentity/gene/324295302': 0.11010401188707279, 'bioentity/gene/325409102': 0.07830609212481426, 'bioentity/gene/324270102': 0.0725111441307578, 'bioentity/gene/326229202': 0.0725111441307578, 'bioentity/gene/324958702': 0.05393759286775631, 'bioentity/gene/323767202': 0.051857355126300145, 'bioentity/gene/324043802': 0.017087667161961365, 'bioentity/gene/324133702': 0.011441307578008915, 'article/pmid/23634996': 0.01114413075780089, 'article/pmid/24120142': 0.009361069836552749, 'bioentity/gene/323805802': 0.008915304606240713, 'bioentity/gene/323568002': 0.008320950965824666, 'bioentity/gene/324295802': 0.008320950965824666, 'article/pmid/21252716': 0.007280832095096582, 'bioentity/gene/324142202': 0.006240713224368499, 'bioentity/gene/321173202': 0.004903417533432392, 'article/pmid/21130701': 0.004606240713224368, 'article/pmid/22522925': 0.004606240713224368, 'bioentity/drug/4140903': 